# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kenzo4k/Flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Plain-Words Contract Definition (5 Answers):

1. **Unit of Analysis (Grain)**: **1 row = 1 pseudonymized content item (`content_hash_id` within `client_hash_id`)** evaluated over a monthly observation period.
2. **Tables Used**:
   - `fact_content_daily_performance` (partitioned by month): daily search metrics (`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`) and GA4 engagement signals.
   - `dim_clients`: client data start dates (`gsc_data_start`, `ga4_data_start`) and cross-client holdout keys.
   - `fact_content_query_90d`: query-level search demand concentration (`visible_queries`, `top_query_share`).
3. **Time Window**:
   - **Observation Window**: Mid-panel month March 2026 (`2026-03-01` to `2026-03-31`, 31 continuous days).
   - **Outcome Window**: Month April 2026 (`2026-04-01` to `2026-04-30`, 30 continuous days).
4. **Target / Proxy to Predict**:
   - `is_declining`: Binary decay indicator where an active content item's search impressions drop by more than 20% month-over-month relative to March ($\text{imp}_{\text{April}} < 0.80 \times \text{imp}_{\text{March}}$).
5. **Deliberately Excluded**:
   - **All April outcome metrics and change ratios** (`april_imp`, `LEAK_decay_ratio`) to eliminate target leakage.
   - **Entity identifiers** (`client_hash_id`, `content_hash_id`) to prevent models from memorizing specific websites.
   - **Redundant tier bins** (e.g. `impression_tier`, `position_tier`) which discretize continuous signals.

In [3]:
# Section 1 Code: Setup, Warehouse Authentication, and Grain/Window Verification
import os, sys, getpass
import duckdb
import pandas as pd
import numpy as np

# Robust token resolution: env var -> Colab Secret -> local .env -> prompt
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    for env_path in ['.env', '../.env', '../../.env']:
        if os.path.exists(env_path):
            try:
                from dotenv import load_dotenv
                load_dotenv(env_path)
                HF_TOKEN = os.environ.get('HF_TOKEN')
                if HF_TOKEN:
                    break
            except Exception:
                pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH_SRC = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL_SRC = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
CLIENTS_SRC = f"read_parquet('{REL}/dim_clients.parquet')"
QUERY_SRC = f"read_parquet('{REL}/fact_content_query_90d.parquet')"

print("=== WAREHOUSE DUCKDB CONNECTION ESTABLISHED ===")
client_count = con.sql(f"SELECT COUNT(*) FROM {CLIENTS_SRC}").fetchone()[0]
print(f" - dim_clients Verified : {client_count} portfolio clients connected")


=== WAREHOUSE DUCKDB CONNECTION ESTABLISHED ===
 - dim_clients Verified : 104 portfolio clients connected


In [4]:
# Section 2 Code: Field Bucket Verification & 5-Feature Frame Creation
feature_cols = ['feat_log_impressions', 'feat_log_clicks', 'feat_avg_position', 'feat_active_days', 'feat_has_ga4']
label_cols = ['is_declining', 'april_imp']
context_cols = ['client_hash_id', 'content_hash_id']
excluded_cols = ['april_imp', 'LEAK_decay_ratio', 'client_hash_id', 'content_hash_id']

feat_leak_overlap = set(feature_cols).intersection(set(excluded_cols))
feat_label_overlap = set(feature_cols).intersection(set(label_cols))

print("=== DATA CONTRACT FIELD BUCKETS ===")
print(f" - Feature Columns (<= 5) : {len(feature_cols)} -> {feature_cols}")
print(f" - Label / Proxy Columns  : {len(label_cols)} -> {label_cols}")
print(f" - Context / Join Keys    : {len(context_cols)} -> {context_cols}")
print(f" - Excluded Columns       : {len(excluded_cols)} -> {excluded_cols}")
print(f"\n=== LEAKAGE & OVERLAP INTEGRITY CHECKS ===")
print(f" - Overlap (Features x Excluded): {list(feat_leak_overlap)} -> {'PASSED (Zero Leakage)' if len(feat_leak_overlap) == 0 else 'FAILED'}")
print(f" - Overlap (Features x Labels)  : {list(feat_label_overlap)} -> {'PASSED (Zero Leakage)' if len(feat_label_overlap) == 0 else 'FAILED'}")

# Load or generate 5-feature sample
cache_candidates = [
    'work/outputs/contract_march_sample.parquet',
    '../outputs/contract_march_sample.parquet',
    '../../outputs/contract_march_sample.parquet'
]
cache_file = next((p for p in cache_candidates if os.path.exists(p)), None)

if cache_file:
    print(f"\nLoading cached sample dataset from: {cache_file}")
    df = pd.read_parquet(cache_file)
else:
    print("\nQuerying warehouse partitions via DuckDB...")
    query = f"""
    WITH march_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_imp,
            SUM(gsc_clicks) AS march_clk,
            AVG(gsc_avg_position) AS march_pos,
            COUNT(DISTINCT report_date) AS march_active_days,
            MAX(CASE WHEN ga4_data_available IS TRUE AND ga4_sessions > 0 THEN 1 ELSE 0 END) AS march_has_ga4
        FROM {MARCH_SRC}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING march_imp >= 50
    ),
    april_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS april_imp
        FROM {APRIL_SRC}
        GROUP BY 1, 2
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        LN(m.march_imp + 1) AS feat_log_impressions,
        LN(m.march_clk + 1) AS feat_log_clicks,
        COALESCE(m.march_pos, 100.0) AS feat_avg_position,
        m.march_active_days AS feat_active_days,
        m.march_has_ga4 AS feat_has_ga4,
        CASE WHEN COALESCE(a.april_imp, 0) < 0.8 * m.march_imp THEN 1 ELSE 0 END AS is_declining,
        (COALESCE(a.april_imp, 0) - m.march_imp) * 1.0 / (m.march_imp + 1) AS LEAK_decay_ratio
    FROM march_agg m
    LEFT JOIN april_agg a
        ON m.client_hash_id = a.client_hash_id
       AND m.content_hash_id = a.content_hash_id
    USING SAMPLE 10000 (reservoir, 42)
    """
    df = con.sql(query).df()
    save_path = 'work/outputs/contract_march_sample.parquet'
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    df.to_parquet(save_path, index=False)

feat_df = df[feature_cols]
print("\n=== FIVE-FEATURE MATRIX SUMMARY ===")
print(f" - Feature Matrix Shape : {feat_df.shape[0]:,} rows x {feat_df.shape[1]} features (Max 5 verified)")
print(f" - Target Base Rate (is_declining): {df['is_declining'].mean()*100:.1f}%")
print("\nDescriptive Statistics:")
print(feat_df.describe().round(2).to_string())


=== DATA CONTRACT FIELD BUCKETS ===
 - Feature Columns (<= 5) : 5 -> ['feat_log_impressions', 'feat_log_clicks', 'feat_avg_position', 'feat_active_days', 'feat_has_ga4']
 - Label / Proxy Columns  : 2 -> ['is_declining', 'april_imp']
 - Context / Join Keys    : 2 -> ['client_hash_id', 'content_hash_id']
 - Excluded Columns       : 4 -> ['april_imp', 'LEAK_decay_ratio', 'client_hash_id', 'content_hash_id']

=== LEAKAGE & OVERLAP INTEGRITY CHECKS ===
 - Overlap (Features x Excluded): [] -> PASSED (Zero Leakage)
 - Overlap (Features x Labels)  : [] -> PASSED (Zero Leakage)

Querying warehouse partitions via DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== FIVE-FEATURE MATRIX SUMMARY ===
 - Feature Matrix Shape : 10,000 rows x 5 features (Max 5 verified)
 - Target Base Rate (is_declining): 52.4%

Descriptive Statistics:
       feat_log_impressions  feat_log_clicks  feat_avg_position  feat_active_days  feat_has_ga4
count              10000.00         10000.00           10000.00          10000.00       10000.0
mean                   6.49             1.00              15.57             27.64           0.5
std                    1.56             1.19              15.61              5.68           0.5
min                    3.93             0.00               0.01              1.00           0.0
25%                    5.19             0.00               5.13             26.00           0.0
50%                    6.36             0.69               9.25             31.00           1.0
75%                    7.63             1.61              20.69             31.00           1.0
max                   12.41             7.83              91

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Per `skills/flyrank/flyrank-data/SKILL.md`, we iterate on a **mid-panel month partition** (`month=2026-03`), not `fact_content_daily_performance_sample`:
> *The sample table is the panel's LAST month (June 2026) — develop label logic there and you are developing inside your own future test window. Treat the final month as a sealed test month.*

We execute three quantitative queries to prove:
1. **The Grain**: 1 row really is `client_hash_id` $\times$ `content_hash_id` $\times$ `report_date` with zero duplicate collisions.
2. **Slice Counts & Date Span**: Total row count and exact date boundaries for March 2026.
3. **Availability Filter (`IS TRUE`)**: Filter on `ga4_data_available IS TRUE` and `gsc_data_available IS TRUE` to show how many rows survive.

In [5]:
# Section 3 Code: Three Warehouse Verification Queries on month=2026-03
print("=== QUERY 1: THE GRAIN PROBE (client x content x date) ===")
dups = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {MARCH_SRC}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f" - Duplicate rows returned: {len(dups)}")
print(f" - Grain verification status: {'PASSED (Zero duplicate rows across entire partition)' if len(dups) == 0 else 'FAILED'}")

print("\n=== QUERY 2: SLICE ROW COUNT & DATE SPAN (month=2026-03) ===")
span_df = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {MARCH_SRC}
""").df()
print(span_df.to_string(index=False))

print("\n=== QUERY 3: AVAILABILITY FILTER (IS TRUE) ===")
avail_df = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_surviving_rows,
        ROUND(COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS ga4_pct_surviving,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_surviving_rows,
        ROUND(COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS gsc_pct_surviving
    FROM {MARCH_SRC}
""").df()
print(avail_df.to_string(index=False))


=== QUERY 1: THE GRAIN PROBE (client x content x date) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 - Duplicate rows returned: 0
 - Grain verification status: PASSED (Zero duplicate rows across entire partition)

=== QUERY 2: SLICE ROW COUNT & DATE SPAN (month=2026-03) ===
 total_rows  distinct_clients  distinct_content   min_date   max_date
    9841378                55            331437 2026-03-01 2026-03-31

=== QUERY 3: AVAILABILITY FILTER (IS TRUE) ===
 total_rows  ga4_surviving_rows  ga4_pct_surviving  gsc_surviving_rows  gsc_pct_surviving
    9841378              413966               4.21             3611061              36.69


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Empirical Reality Checks & The Warehouse Leakage Trap

1. **Unbalanced Portfolio Depths**: Client tracking histories start at different dates (`dim_clients.gsc_data_start`), requiring cross-client Group splits rather than standard random row splits.
2. **Zero-Fill Gotcha**: Rows prior to `ga4_data_start` have zeros with `ga4_data_available = FALSE`. Filtering on `IS TRUE` prevents treating missing instrumentation as zero engagement.
3. **The Leakage Trap (Real Warehouse Data)**: Demonstrating the leakage lesson from Notebook 02 on warehouse data:
   - Train honest baseline model on 5 pre-decision features.
   - Deliberately inject **ONE** label-derived feature (`LEAK_decay_ratio = (april_imp - march_imp) / (march_imp + 1)`).
   - Watch the classifier score jump toward 1.00 ROC-AUC.
   - Delete the trap column and retain the honest baseline number.

In [6]:
# Section 4 Code: Portfolio Boundaries & The Leakage Trap Experiment
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# 1. Portfolio history check
client_history = con.sql(f"""
    SELECT
        COUNT(*) AS total_clients,
        MIN(gsc_data_start) AS earliest_gsc,
        MAX(gsc_data_start) AS latest_gsc,
        COUNT(*) FILTER (WHERE ga4_data_start IS NOT NULL) AS clients_with_ga4
    FROM {CLIENTS_SRC}
""").df()
print("=== PORTFOLIO HISTORY BOUNDARIES ===")
print(client_history.to_string(index=False))

# 2. The Leakage Trap
X_honest = df[feature_cols]
y = df['is_declining']
leak_col = df['LEAK_decay_ratio']

X_tr, X_te, y_tr, y_te, leak_tr, leak_te = train_test_split(
    X_honest, y, leak_col, test_size=0.25, random_state=42, stratify=y
)

# Fit Honest Baseline Model
clf_honest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf_honest.fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, clf_honest.predict_proba(X_te)[:, 1])

# The Trap: Add ONE label-derived column on purpose
X_tr_leaky = X_tr.copy()
X_tr_leaky['LEAK_decay_ratio'] = leak_tr
X_te_leaky = X_te.copy()
X_te_leaky['LEAK_decay_ratio'] = leak_te

clf_leaky = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf_leaky.fit(X_tr_leaky, y_tr)
leaky_auc = roc_auc_score(y_te, clf_leaky.predict_proba(X_te_leaky)[:, 1])

print("\n=== LEAKAGE EXPERIMENT RESULTS (Real Warehouse Data) ===")
print(f" - Honest Model ROC-AUC : {honest_auc:.4f}")
print(f" - Leaky Model ROC-AUC  : {leaky_auc:.4f} (ARTIFICIAL JUMP TO 1.00 DETECTED!)")
print(f" - Leakage Jump Delta   : +{leaky_auc - honest_auc:.4f}")

# Delete the leaky column and retain honest score
del X_tr_leaky['LEAK_decay_ratio']
del X_te_leaky['LEAK_decay_ratio']
print("\n[Action Taken]: Deleted 'LEAK_decay_ratio' from feature set. Retaining honest score:", round(honest_auc, 4))


=== PORTFOLIO HISTORY BOUNDARIES ===
 total_clients earliest_gsc latest_gsc  clients_with_ga4
           104   2025-01-27 2026-06-02                51

=== LEAKAGE EXPERIMENT RESULTS (Real Warehouse Data) ===
 - Honest Model ROC-AUC : 0.6772
 - Leaky Model ROC-AUC  : 1.0000 (ARTIFICIAL JUMP TO 1.00 DETECTED!)
 - Leakage Jump Delta   : +0.3228

[Action Taken]: Deleted 'LEAK_decay_ratio' from feature set. Retaining honest score: 0.6772


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.